# Mini Project 1 — Analysis Notebook

**Your name:** Ashlesha Hadkar 
**Dataset:** Week 6 / `KCC Data /combined_kcc_3states_en.csv` (merged Maharashtra, Karnataka, Uttar Pradesh KCC exports; produced by `combine_translate_kcc.py`).
**Date:**  6/05/2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Original source:** data.gov.in
https://api.data.gov.in/resource/cef25fe2-9231-4128-8aec-2c948fedd43f

**Data accessed from:** State-level KCC CSV folders under `Week 6/KCC Data /` (Maharashtra, Karnataka, Uttar Pradesh), combined with `combine_translate_kcc.py`, which also adds `KccAns_en` for translated answers when translation has been run.

**Why this dataset**

Farmer support systems like KCC are designed to  bring agricultural expertise directly to rural communities, but design intent and ground-level reality are not always the same thing — and that gap is a core HCD question. This dataset is a lens for examining 
who gets helped and who doesn't, how underserved communities interact with government-built technology, and whether a public support system is actually reaching the people it was built for.

**Three analytical questions:**

1. Which types of queries dominate each month, and how does the mix change across the year?
2. Are queries tagged to the right QueryType? A mismatch audit.
3. What share of queries are for Agronomic / Crop Advisory support, how does the topic breakdown differ by state, and is demand growing?

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [2]:
# Load combined KCC CSV (built by combine_translate_kcc.py → KCC Data /combined_kcc_3states_en.csv)
from pathlib import Path

filename = 'combined_kcc_3states_en.csv'
abs_csv = Path('/Users/ashleshahadkar/Documents/HCDE 530 Repository/Week 6/KCC Data /combined_kcc_3states_en.csv')
kcc_dir = Path('KCC Data ')

# Absolute-first path lookup avoids failures when notebook cwd changes.
candidates = [
    abs_csv,
    Path.cwd() / kcc_dir / filename,
    Path.cwd() / Path('Week 6') / kcc_dir / filename,
    Path.cwd().parent / Path('Week 6') / kcc_dir / filename,
    Path.cwd().parent.parent / Path('Week 6') / kcc_dir / filename,
]
# This code finds the path to the CSV file for the KCC CSVs.
csv_path = next((path for path in candidates if path.exists()), None)
if csv_path is None:
    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find '{filename}'. Run from Week 6/: python3 combine_translate_kcc.py (or --resume). Checked:\n{searched}"
    )
# This code reads the CSV file for the KCC CSVs.
df = pd.read_csv(csv_path, encoding='utf-8-sig', low_memory=False)

# Drop Maharashtra 2025 rows so all three states cover the same 2022-2024 window
df = df[~((df['StateName'].astype(str).str.strip() == 'MAHARASHTRA') & (df['year'] == 2025))]
df = df.reset_index(drop=True)

df['QueryType_clean'] = df['QueryType'].astype(str).str.strip()

print(f'Loaded from: {csv_path}')
print(df.shape)
df.head()

Loaded from: /Users/ashleshahadkar/Documents/HCDE 530 Repository/Week 6/KCC Data /combined_kcc_3states_en.csv
(5400, 18)


,BlockName,Category,CreatedOn,Crop,DistrictName,KCCCallID,KccAns,QueryText,QueryType,Season,Sector,StateName,day,month,year,_source_file,KccAns_en,QueryType_clean
0,MADHA,Sugar and Starch Crops,2022-04-24T17:01:29.06,Sugarcane (Noble Cane),SOLAPUR,1197635,ऊस पिकासाठी-१२ ६१ ०० ६० ग्रॅम + मायक्रोला ३० म...,FARMER ASKED ABOUT FERTILIZER SPRAY ON SUGARCA...,Fertilizer Use and Availability,NaN,AGRICULTURE,MAHARASHTRA,24,4,2022,Maharashtra/2022/apr.csv,For sugarcane crop- 12 61 00 60 gm + micro wit...,Fertilizer Use and Availability
1,AUNDHA NAGNATH,Condiments and Spices,2022-04-24T17:02:24.237,Turmeric,HINGOLI,1197668,हवामान विभागाच्या अंदाजानुसार आपल्या औंढा-नागन...,Farmer asked query on Weather,Weather,NaN,HORTICULTURE,MAHARASHTRA,24,4,2022,Maharashtra/2022/apr.csv,According to the forecast of the Meteorologica...,Weather
2,JAMKHED,Others,2022-04-25T09:01:04.307,Others,AHMADNAGAR,1200438,⦁\tप्रधानमंत्री किसान सन्मान निधी योजना संबंधी...,Farmer asked about PM Kisan Sanman Nidhi Yojan...,Government Schemes,NaN,AGRICULTURE,MAHARASHTRA,25,4,2022,Maharashtra/2022/apr.csv,⦁ For more information regarding Prime Ministe...,Government Schemes
3,PAUNI,Others,2022-04-25T09:05:43.497,Others,BHANDARA,1200546,पी एम किसान साठी तुमच्या जवळील तहसील ऑफिस ला ज...,FARMER ASKED ABOUT PM KISAN ?,Government Schemes,NaN,AGRICULTURE,MAHARASHTRA,25,4,2022,Maharashtra/2022/apr.csv,For PM Kisan visit your nearest tehsil office ...,Government Schemes
4,MANGALVEDHE,Others,2022-04-25T09:16:57.56,Others,SOLAPUR,1200431,"हवामान विभागाच्या अंदाजानुसार,आपल्या तालुक्यात...",Farmer asked query on Weather,Weather,NaN,AGRICULTURE,MAHARASHTRA,25,4,2022,Maharashtra/2022/apr.csv,According to the forecast of Meteorological De...,Weather


In [3]:
# Check column types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5400 entries, 0 to 5399
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   BlockName        5400 non-null   object 
 1   Category         5400 non-null   object 
 2   CreatedOn        5400 non-null   object 
 3   Crop             5400 non-null   object 
 4   DistrictName     5400 non-null   object 
 5   KCCCallID        5400 non-null   int64  
 6   KccAns           5390 non-null   object 
 7   QueryText        5400 non-null   object 
 8   QueryType        5400 non-null   object 
 9   Season           0 non-null      float64
 10  Sector           5400 non-null   object 
 11  StateName        5400 non-null   object 
 12  day              5400 non-null   int64  
 13  month            5400 non-null   int64  
 14  year             5400 non-null   int64  
 15  _source_file     5400 non-null   object 
 16  KccAns_en        5390 non-null   object 
 17  QueryType_clea

In [4]:
# Summary statistics for numeric columns
df.describe()

,KCCCallID,Season,day,month,year
count,5.400000e+03,0.0,5400.000000,5400.000000,5400.000000
mean,3.125479e+06,NaN,14.819074,6.500000,2023.000000
std,2.079564e+06,NaN,8.347631,2.500232,0.816572
min,5.557060e+05,NaN,1.000000,3.000000,2022.000000
25%,1.337276e+06,NaN,8.000000,4.000000,2022.000000
50%,2.589199e+06,NaN,14.000000,6.500000,2023.000000
75%,4.927158e+06,NaN,23.000000,9.000000,2024.000000
max,7.139959e+06,NaN,31.000000,10.000000,2024.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1: What query types are farmers asking across Maharashtra, Karnataka, and Uttar Pradesh?**

Overall distribution across all three states to set the context.

**Interpretation**

Weather, government schemes, plant protection and market information are the top categories with highest number of requests for information. 

In [5]:
# Q1: Overall query-type distribution across Maharashtra, Karnataka, and Uttar Pradesh
q1 = df.copy()
q1['QueryType_clean'] = q1['QueryType'].astype(str).str.strip()

q1_dist = (
    q1['QueryType_clean']
    .value_counts()
    .reset_index()
)
q1_dist.columns = ['QueryType', 'count']
q1_dist['pct'] = 100 * q1_dist['count'] / q1_dist['count'].sum()

print('Overall distribution of query types across all three states:')
display(q1_dist)

fig_q1 = px.bar(
    q1_dist,
    x='QueryType',
    y='count',
    title='Q1: Query types farmers ask most often across 3 states',
    labels={'count': 'Number of queries', 'QueryType': 'Query type'},
    text=q1_dist['pct'].round(1).astype(str) + '%',
)
fig_q1.update_layout(xaxis_tickangle=-45)
fig_q1.update_traces(textposition='outside')
fig_q1.show()

Overall distribution of query types across all three states:


,QueryType,count,pct
0,Weather,2317,42.907407
1,Government Schemes,1100,20.370370
2,Plant Protection,637,11.796296
3,Market Information,273,5.055556
4,Nutrient Management,208,3.851852
5,Cultural Practices,183,3.388889
6,Fertilizer Use and Availability,146,2.703704
7,Weed Management,100,1.851852
8,Varieties,67,1.240741
9,Water Management,66,1.222222


**Question 2: For each state separately, how have all query types trended from 2022 to 2024 — are certain issues increasing, decreasing, or stable over time?**

Three separate line charts, one per state.

**Interpretation for Q2:**  
*(Which query types are growing, shrinking, or stable in each state?)*

In [6]:
# Q2: State-wise query-type trends (2022-2024), with top types + Other for readability
q2 = df.copy()
q2['StateName'] = q2['StateName'].astype(str).str.strip()
q2['QueryType_clean'] = q2['QueryType'].astype(str).str.strip()
q2 = q2[q2['year'].between(2022, 2024)].copy()

top_types_q2 = q2['QueryType_clean'].value_counts().head(8).index.tolist()
q2['QueryType_grouped'] = q2['QueryType_clean'].where(q2['QueryType_clean'].isin(top_types_q2), 'Other')

q2_trend = (
    q2.groupby(['StateName', 'year', 'QueryType_grouped'])
    .size()
    .reset_index(name='count')
    .sort_values(['StateName', 'year', 'count'], ascending=[True, True, False])
)

print('Q2 trend table (2022-2024, top query types + Other):')
display(q2_trend)

for state in sorted(q2_trend['StateName'].unique()):
    state_data = q2_trend[q2_trend['StateName'] == state]
    fig_state = px.line(
        state_data,
        x='year',
        y='count',
        color='QueryType_grouped',
        markers=True,
        title=f'Q2: {state} query-type trend (2022-2024)',
        labels={'year': 'Year', 'count': 'Number of queries', 'QueryType_grouped': 'Query type'},
    )
    fig_state.update_layout(xaxis=dict(dtick=1), legend_title='Query type')
    fig_state.show()

Q2 trend table (2022-2024, top query types + Other):


,StateName,year,QueryType_grouped,count
6,KARNATAKA,2022,Weather,319
1,KARNATAKA,2022,Government Schemes,88
2,KARNATAKA,2022,Market Information,53
5,KARNATAKA,2022,Plant Protection,43
3,KARNATAKA,2022,Nutrient Management,41
...,...,...,...,...
71,UTTAR PRADESH,2024,Cultural Practices,31
75,UTTAR PRADESH,2024,Nutrient Management,26
72,UTTAR PRADESH,2024,Fertilizer Use and Availability,11
79,UTTAR PRADESH,2024,Weed Management,11


**Cultural Practices categorization audit (targeted check):**

This diagnostic checks rows currently tagged as `Cultural Practices` and infers an expected category from query keywords. Rows where expected category is not Cultural Practices are flagged as potential misclassification.

In [7]:
# Cultural Practices misclassification audit
import re

cp = df[df['QueryType_clean'] == 'Cultural Practices'].copy()
cp['query_lower'] = cp['QueryText'].astype(str).str.lower()

def infer_expected_category(text):
    if re.search(r'scheme|yojana|subsid|pm[- ]?kisan|insurance|credit|loan|kcc|registration|portal', text):
        return 'Government Schemes / Admin'
    if re.search(r'pest|insect|worm|caterpillar|disease|blight|rot|fung|spray|pesticide', text):
        return 'Plant Protection'
    if re.search(r'fertili|urea|dap|npk|potash|manure|nutrient|micronutrient|zinc', text):
        return 'Fertilizer / Nutrient'
    if re.search(r'irrigat|water|drip|sprinkler|canal|moisture', text):
        return 'Water / Irrigation'
    if re.search(r'market|price|rate|mandi|sell|buyer|msp', text):
        return 'Market / Price'
    if re.search(r'seed|variet|sowing|spacing|pruning|nursery|transplant|cultivation|harvest', text):
        return 'Cultural Practices'
    return 'Other / Unclear'

cp['expected_category'] = cp['query_lower'].apply(infer_expected_category)
cp['current_tag'] = 'Cultural Practices'
cp['potential_misclassified'] = cp['expected_category'] != 'Cultural Practices'

total_cp = len(cp)
flagged_count = int(cp['potential_misclassified'].sum())
print(f'Cultural Practices rows checked: {total_cp}')
print(f'Potentially misclassified rows: {flagged_count} ({100*flagged_count/max(total_cp,1):.1f}%)')

expected_order = [
    'Cultural Practices',
    'Plant Protection',
    'Fertilizer / Nutrient',
    'Water / Irrigation',
    'Market / Price',
    'Government Schemes / Admin',
    'Other / Unclear',
]

heat = (
    cp.groupby(['expected_category', 'current_tag'])
    .size()
    .reset_index(name='count')
)
heat['expected_category'] = pd.Categorical(heat['expected_category'], categories=expected_order, ordered=True)
heat = heat.sort_values('expected_category')

fig_cp = px.density_heatmap(
    heat,
    x='current_tag',
    y='expected_category',
    z='count',
    color_continuous_scale='YlOrRd',
    text_auto=True,
    title='Potential Cultural Practices misclassification: Expected vs Current tag',
    labels={'current_tag': 'Current QueryType tag', 'expected_category': 'Expected category (keyword-based)', 'count': 'Rows'},
)
fig_cp.update_layout(yaxis_title='Expected category (keyword-based)', xaxis_title='Current QueryType tag')
fig_cp.show()

flagged_rows = cp[cp['potential_misclassified']].copy()
show_cols = ['StateName', 'DistrictName', 'Crop', 'QueryText', 'expected_category']

print(f'\nFlagged rows count: {len(flagged_rows)}')
print('Top 20 flagged row texts and details:')
display(flagged_rows[show_cols].head(20).reset_index(drop=True))


Cultural Practices rows checked: 183
Potentially misclassified rows: 145 (79.2%)



Flagged rows count: 145
Top 20 flagged row texts and details:


,StateName,DistrictName,Crop,QueryText,expected_category
0,MAHARASHTRA,KOLHAPUR,Brinjal,FARMER ASKED ABOUT INFORMATION OF BRINJAL CROP ?,Other / Unclear
1,MAHARASHTRA,PUNE,Brinjal,FARMER ASKED ABOUT VARITIES OF BRIJAL ?,Other / Unclear
2,MAHARASHTRA,AHMADNAGAR,Pumpkin,farmer ask about duration of pumpkin crop,Other / Unclear
3,MAHARASHTRA,NAGPUR,Pigeon pea (red gram/arhar/tur),FARMER ASKED ABOUT HELPLINE NO OF KRUSHI VIDYA...,Other / Unclear
4,MAHARASHTRA,AHMADNAGAR,Sorghum (Jowar/Great Millet),FARMER ASKED ABOUT SOEING TIME OF JOWAR ?,Other / Unclear
5,MAHARASHTRA,JALGAON,Banana,Asked about cultural practices in custard apple?,Other / Unclear
6,MAHARASHTRA,DHULE,Brinjal,farmer ask about information about mulching pa...,Other / Unclear
7,MAHARASHTRA,SOLAPUR,Banana,FARMER ASKED ABOUT RIPENING OF BANANA FRUITS?,Other / Unclear
8,MAHARASHTRA,SOLAPUR,Soybean (bhat),FARMER ASKED ABOUT YIELD OF SOYABEAN CROP ?,Other / Unclear
9,MAHARASHTRA,SOLAPUR,Soybean (bhat),FARMER ASKED ABOUT CULTURAL PRACTICES IN SOYBE...,Other / Unclear


**Question 3: Which districts have the lowest call volume — and what are those farmers asking about?**

Also compare with the top 5 districts in a separate visual.

**Interpretation for Q3:**  
*(What does low volume potentially mean for service access? How are top vs bottom districts different in topic demand?)*

In [8]:
# Q3: Lowest-volume districts with all-district context treemap
q3 = df.copy()
q3['StateName'] = q3['StateName'].astype(str).str.strip()
q3['DistrictName'] = q3['DistrictName'].astype(str).str.strip()
q3['district_label'] = q3['DistrictName'] + ' (' + q3['StateName'] + ')'

district_counts = (
    q3.groupby(['StateName', 'DistrictName', 'district_label'])
    .size()
    .reset_index(name='count')
)

bottom5 = district_counts.nsmallest(5, 'count').copy()
print('Bottom 5 districts by call volume:')
display(bottom5[['district_label', 'count']])

fig_bottom = px.bar(
    bottom5.sort_values('count', ascending=True),
    x='count',
    y='district_label',
    orientation='h',
    title='Q3A: Bottom 5 districts by call volume',
    labels={'count': 'Number of queries', 'district_label': 'District (State)'},
)
fig_bottom.show()

fig_tree = px.treemap(
    district_counts,
    path=['StateName', 'DistrictName'],
    values='count',
    title='Q3B: All districts call-volume context (smaller boxes indicate lower volume)',
)
fig_tree.update_traces(textinfo='label+value')
fig_tree.show()


Bottom 5 districts by call volume:


,district_label,count
55,RAIGARH (MAHARASHTRA),1
56,RATNAGIRI (MAHARASHTRA),1
64,9999 (UTTAR PRADESH),1
19,KODAGU (KARNATAKA),2
52,PALGHAR (MAHARASHTRA),2


**Question 4: Across all three states, which query type is most dominant — ranked in descending order?**

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [9]:
# Q4: Dominant query types ranked in descending order across all three states
q4 = df.copy()
q4['QueryType_clean'] = q4['QueryType'].astype(str).str.strip()

q4_rank = (
    q4['QueryType_clean']
    .value_counts()
    .reset_index()
)
q4_rank.columns = ['QueryType', 'count']

print('Q4 ranked query-type counts (all states combined):')
display(q4_rank)

fig_q4 = px.bar(
    q4_rank,
    x='QueryType',
    y='count',
    title='Q4: Dominant query types across Maharashtra, Karnataka, and Uttar Pradesh',
    labels={'QueryType': 'Query type', 'count': 'Number of queries'},
)
fig_q4.update_layout(xaxis_tickangle=-45)
fig_q4.show()


Q4 ranked query-type counts (all states combined):


,QueryType,count
0,Weather,2317
1,Government Schemes,1100
2,Plant Protection,637
3,Market Information,273
4,Nutrient Management,208
5,Cultural Practices,183
6,Fertilizer Use and Availability,146
7,Weed Management,100
8,Varieties,67
9,Water Management,66


**Question 5: How does demand for Government Schemes information compare to Agronomy advice (Plant Protection, Fertilizer, Nutrient Management) across districts?**

Are farmers calling more for bureaucratic help or technical farming help?

In [10]:
# Q5: District-level Government Schemes vs Agronomy demand (scatter)
AGRONOMY_CORE = {
    'Plant Protection',
    'Fertilizer Use and Availability',
    'Nutrient Management',
}

q5 = df.copy()
q5['StateName'] = q5['StateName'].astype(str).str.strip()
q5['DistrictName'] = q5['DistrictName'].astype(str).str.strip()
q5['QueryType_clean'] = q5['QueryType'].astype(str).str.strip()
q5['district_label'] = q5['DistrictName'] + ' (' + q5['StateName'] + ')'

q5['demand_group'] = q5['QueryType_clean'].apply(
    lambda qt: 'Government Schemes' if qt == 'Government Schemes'
    else ('Agronomy Advice' if qt in AGRONOMY_CORE else 'Other')
)
q5_focus = q5[q5['demand_group'].isin(['Government Schemes', 'Agronomy Advice'])].copy()

district_counts = (
    q5_focus.groupby(['district_label', 'StateName', 'demand_group'])
    .size()
    .reset_index(name='count')
)

q5_wide = (
    district_counts.pivot_table(
        index=['district_label', 'StateName'],
        columns='demand_group',
        values='count',
        fill_value=0,
    )
    .reset_index()
)
q5_wide.columns.name = None

if 'Government Schemes' not in q5_wide.columns:
    q5_wide['Government Schemes'] = 0
if 'Agronomy Advice' not in q5_wide.columns:
    q5_wide['Agronomy Advice'] = 0

q5_wide['total_focus'] = q5_wide['Government Schemes'] + q5_wide['Agronomy Advice']
q5_plot = q5_wide.nlargest(40, 'total_focus').copy()

print('Top districts in Q5 comparison (preview of first 10 rows):')
display(q5_plot[['district_label', 'StateName', 'Government Schemes', 'Agronomy Advice', 'total_focus']].head(10))
print('Scatter plot below:')

fig_q5 = px.scatter(
    q5_plot,
    x='Government Schemes',
    y='Agronomy Advice',
    color='StateName',
    hover_name='district_label',
    size='total_focus',
    title='Q5: District skew between Government Schemes and Agronomy demand',
    labels={
        'Government Schemes': 'Government Schemes query count',
        'Agronomy Advice': 'Agronomy query count',
        'StateName': 'State',
    },
)
fig_q5.add_shape(type='line', x0=0, y0=0, x1=q5_plot['Government Schemes'].max(), y1=q5_plot['Government Schemes'].max(), line=dict(dash='dash'))
fig_q5.show()


Top districts in Q5 comparison (preview of first 10 rows):


,district_label,StateName,Government Schemes,Agronomy Advice,total_focus
38,CHITRADURGA (KARNATAKA),KARNATAKA,63.0,32.0,95.0
137,YEVATMAL (MAHARASHTRA),MAHARASHTRA,13.0,70.0,83.0
68,JALGAON (MAHARASHTRA),MAHARASHTRA,30.0,52.0,82.0
82,KOPPAL (KARNATAKA),KARNATAKA,47.0,19.0,66.0
12,BADAUN (UTTAR PRADESH),UTTAR PRADESH,45.0,19.0,64.0
55,GHAZIPUR (UTTAR PRADESH),UTTAR PRADESH,35.0,28.0,63.0
25,BELGAUM (KARNATAKA),KARNATAKA,31.0,27.0,58.0
102,OSMANABAD (MAHARASHTRA),MAHARASHTRA,37.0,20.0,57.0
16,BALLIA (UTTAR PRADESH),UTTAR PRADESH,33.0,15.0,48.0
29,BIJAPUR (KARNATAKA),KARNATAKA,27.0,21.0,48.0


Scatter plot below:


**Question 6: Which crops are generating the most technical queries (Plant Protection, Fertilizer Use, Nutrient Management) — and does this differ by state?**

This helps identify where KCC may need crop-specific specialist support.

In [11]:
# Q6: Technical queries by crop and state (heatmap)
TECH_TYPES = {
    'Plant Protection',
    'Fertilizer Use and Availability',
    'Nutrient Management',
}

q6 = df.copy()
q6['StateName'] = q6['StateName'].astype(str).str.strip()
q6['QueryType_clean'] = q6['QueryType'].astype(str).str.strip()
q6['Crop_clean'] = q6['Crop'].astype(str).str.strip()

q6 = q6[
    q6['QueryType_clean'].isin(TECH_TYPES)
    & q6['Crop_clean'].notna()
    & (q6['Crop_clean'] != '')
    & (q6['Crop_clean'].str.lower() != 'nan')
].copy()

crop_state = (
    q6.groupby(['Crop_clean', 'StateName'])
    .size()
    .reset_index(name='count')
)

top_crops = crop_state.groupby('Crop_clean')['count'].sum().sort_values(ascending=False).head(20).index.tolist()
crop_state_top = crop_state[crop_state['Crop_clean'].isin(top_crops)].copy()

print('Q6 crop x state counts (top 20 crops by total technical-query volume):')
display(crop_state_top.sort_values(['count'], ascending=False))

fig_q6 = px.density_heatmap(
    crop_state_top,
    x='StateName',
    y='Crop_clean',
    z='count',
    color_continuous_scale='Viridis',
    title='Q6: Technical query concentration by crop and state',
    labels={'StateName': 'State', 'Crop_clean': 'Crop', 'count': 'Technical query count'},
    text_auto=True,
)
fig_q6.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_q6.show()


Q6 crop x state counts (top 20 crops by total technical-query volume):


,Crop_clean,StateName,count
50,Cotton (Kapas),MAHARASHTRA,108
108,Paddy (Dhan),UTTAR PRADESH,89
134,Soybean (bhat),MAHARASHTRA,49
138,Sugarcane (Noble Cane),UTTAR PRADESH,38
86,Mango,UTTAR PRADESH,34
137,Sugarcane (Noble Cane),MAHARASHTRA,30
2,Arecanut,KARNATAKA,24
100,Onion,MAHARASHTRA,23
49,Cotton (Kapas),KARNATAKA,22
36,Chillies,KARNATAKA,21


---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.